# Links 
github repo: https://github.com/vildeskobba/IND320vilde 

Streamlit app: https://ind320vilde.streamlit.app/ 

Panopto_video: 


In [127]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [128]:
import pandas as pd
import requests
import tomllib
from pymongo import MongoClient
import certifi
import numpy as np
from river import preprocessing, linear_model, compose
import geopandas as gpd


In [129]:
def get_exchange_rates(url:str):
    response = requests.get(
        url,
        params = {
            "format": "sdmx-json",
            "locale": "no",
        },
        timeout = 30,
    )
    response.raise_for_status() #raise error if request was unsuccessfull 
    data = response.json()["data"] # convert json to dict

    dimensions = data["structure"]["dimensions"]
    series_dims = dimensions["series"]

    currencies = {
        d["id"]: d["values"] for d in series_dims
    }

    dates = dimensions["observation"][0]["values"]

    records = []
    for key, series in data["dataSets"][0]["series"].items():
        indices = key.split(":")
        base = currencies["BASE_CUR"][int(indices[2])]["id"]
        quote = currencies["QUOTE_CUR"][int(indices[3])]["id"]

        for date, values in series["observations"].items():
            records.append([
                dates[int(date)]["id"], base, quote, values[0]
            ])

    df = pd.DataFrame(
        records,
        columns=["TIME_PERIOD", "BASE_CUR", "QUOTE_CUR", "OBS_VALUE"]
        )

    df["TIME_PERIOD"] = pd.to_datetime(df["TIME_PERIOD"]) 

    return df
    

In [130]:
usd_rates_df = get_exchange_rates("https://data.norges-bank.no/api/data/EXR/B.USD.NOK.SP?endPeriod=2026-11-17&format=sdmx-json&locale=no&startPeriod=2024-01-01")

print(usd_rates_df.head())

  TIME_PERIOD BASE_CUR QUOTE_CUR OBS_VALUE
0  2024-01-02      USD       NOK   10.2971
1  2024-01-03      USD       NOK   10.3672
2  2024-01-04      USD       NOK   10.3027
3  2024-01-05      USD       NOK   10.3553
4  2024-01-08      USD       NOK     10.38


In [131]:
eur_rates_df = get_exchange_rates("https://data.norges-bank.no/api/data/EXR/B.EUR.NOK.SP?endPeriod=2026-11-17&format=sdmx-json&locale=no&startPeriod=2024-01-01")

print(eur_rates_df.head())

  TIME_PERIOD BASE_CUR QUOTE_CUR OBS_VALUE
0  2024-01-02      EUR       NOK   11.2815
1  2024-01-03      EUR       NOK     11.32
2  2024-01-04      EUR       NOK   11.2845
3  2024-01-05      EUR       NOK    11.309
4  2024-01-08      EUR       NOK    11.362


In [132]:


with open(".streamlit/secrets.toml", "rb") as f:
    secrets = tomllib.load(f)

mongo_uri = secrets["MONGO_URI"]

client = MongoClient(mongo_uri, tlsCAFile=certifi.where())

db = client["power_data"]

In [133]:
# magasin statestikk 
from utils.APIs.NVE_API import get_nve_data
magasinstatistikk_df = get_nve_data(start_date="2024.01.01", end_date="2024.12.31")
print(f"magasinstatistikk, shape: {magasinstatistikk_df.shape} \n", magasinstatistikk_df[["date", "reservoir level", "area number"]].head(4), "\n")

# entso-e 
with open(".streamlit/secrets.toml", "rb") as f:
    secrets = tomllib.load(f)
mongo_uri = secrets["MONGO_URI"]
client = MongoClient(mongo_uri, tlsCAFile=certifi.where())
db = client["power_data"]
daily_average_flows_entsoe_df = pd.DataFrame(
    list(db["daily_average_flows"].find({}, {"_id": 0}))
)
print(f"entso-e, shape: {daily_average_flows_entsoe_df.shape} \n", daily_average_flows_entsoe_df.head(4), "\n")

# openmeteo 
open_meteo_df = pd.read_parquet("data/open_meteo/NO_1.parquet")
print(f"open meteo, shape: {open_meteo_df.shape} \n", open_meteo_df.head(4), "\n")

# norges bank 
exchange_rates_df = pd.concat([eur_rates_df, usd_rates_df], ignore_index=True) 
print(f"norges bank, shape: {exchange_rates_df.shape} \n", exchange_rates_df.head(4))

magasinstatistikk, shape: (468, 11) 
         date  reservoir level  area number
0 2024-01-07         0.621008            2
1 2024-01-07         0.527503            4
2 2024-01-07         0.635426            1
3 2024-01-07         0.597728            5 

entso-e, shape: (10234, 4) 
    timestamp from_zone to_zone   flow_mw
0 2024-10-18      SE_3    NO_1   849.540
1 2024-10-18      NO_1    SE_3     0.000
2 2024-10-19      NO_2    DK_1  1631.475
3 2024-10-19      NO_3    SE_2     0.000 

open meteo, shape: (8784000, 5) 
                        time   latitude  longitude  temperature  precipitation
0 2024-01-01 00:00:00+01:00  61.936407  10.919136    -8.735001            0.1
1 2024-01-01 01:00:00+01:00  61.936407  10.919136    -8.635000            0.1
2 2024-01-01 02:00:00+01:00  61.936407  10.919136    -8.635000            0.1
3 2024-01-01 03:00:00+01:00  61.936407  10.919136    -8.535000            0.1 

norges bank, shape: (1448, 4) 
   TIME_PERIOD BASE_CUR QUOTE_CUR OBS_VALUE
0  2024-

time zone differences for norges bank, open-meteo, entso-e and magasin statistikk: 
I asked chat gpt, and got this answer: 
| Data source | Time convention | Resolution | Important detail |
|---|---|---|---|
| Norges Bank | CET/CEST | Daily | Exchange rates published around 16:00 local time |
| Open-Meteo | Configurable | Hourly/daily | Default is GMT/UTC; can request Europe/Oslo |
| ENTSO-E | UTC in API responses | Hourly/15-min | Daily periods follow the relevant bidding area's local calendar |
| NVE | Norwegian local calendar | Weekly | Measurements represent Sunday at 24:00, or the nearest available measurement |

it provided links as sources and i checked them all and found: 
 for norges bank, data is published daily at 16:00(assuming local time), the data shows it is collected at "ECB concertation tidspunkt 14:15 CET", central european time. 

 open meteo: i set time to Oslo-timezone in the api func. resolution is hourly, but im guessing i will use daily, calculating a daily average later in assignment. 

 entso-e: daily resolution 

 nve: weekly, confirmed on the website. 

 to be able to use them all im going to get all the time cols on the same format. oslo and brussels have no time difference, both on CET time, im not going to account for daylight savings, becuase i will use daily resolution. just for my own ease, i will rename the time col to "date" for all the dfs. 

In [134]:
magasinstatistikk_df = magasinstatistikk_df[
    magasinstatistikk_df["area type"] == "EL"
].copy() 

magasinstatistikk_df["area"] = (
    "NO_" + magasinstatistikk_df["area number"].astype(str)
)

magasinstatistikk_df = magasinstatistikk_df[[
    "year (ISO)", 
    "week (ISO)", 
    "area", 
    "reservoir level", 
    "level (TWh)", 
    "capacity (TWh)"
    ]].copy()

magasinstatistikk_df = magasinstatistikk_df[[
    "year (ISO)", "week (ISO)", "area",
    "reservoir level", "level (TWh)", "capacity (TWh)"
]].rename(columns={
    "year (ISO)": "iso_year",
    "week (ISO)": "iso_week",
    "area": "area",
    "reservoir level": "reservoir_level",
    "level (TWh)": "level_TWh",
    "capacity (TWh)": "capacity_TWh"
}).copy()


In [135]:
# Helper function: convert dates to ISO year and ISO week
def add_iso_week(df, date_col="date"):
    df = df.copy()
    dates = pd.to_datetime(df[date_col])

    iso = dates.dt.isocalendar()
    df["iso_year"] = iso["year"]
    df["iso_week"] = iso["week"]

    return df

In [136]:
# ENTSO-E: UTC -> Norwegian time
daily_average_flows_entsoe_df["date"] = (
    pd.to_datetime(
        daily_average_flows_entsoe_df["timestamp"],
        utc=True
    )
    .dt.tz_convert("Europe/Oslo")
    .dt.tz_localize(None)
)

# Aggregate to weekly averages
entsoe_weekly = (
    daily_average_flows_entsoe_df
    .groupby(pd.to_datetime(
        daily_average_flows_entsoe_df["date"]
    ).dt.to_period("W-SUN"))
    .agg({"flow_mw": "mean"})
    .reset_index(drop=True)
)

# Add ISO year/week
entsoe_weekly["date"] = (
    daily_average_flows_entsoe_df
    .groupby(pd.to_datetime(
        daily_average_flows_entsoe_df["date"]
    ).dt.to_period("W-SUN"))["date"]
    .first()
    .values
)

daily_average_flows_entsoe_df = add_iso_week(entsoe_weekly)


In [137]:
# Open-Meteo: hourly -> weekly
open_meteo_df["date"] = (
    pd.to_datetime(open_meteo_df["time"])
    .dt.tz_localize(None)
)

open_meteo_df["week"] = (
    open_meteo_df["date"].dt.to_period("W-SUN")
)

weather_weekly = (
    open_meteo_df
    .groupby(["latitude", "longitude", "week"])
    .agg({
        "temperature": "mean",
        "precipitation": "sum"
    })
    .reset_index()
)

weather_weekly["date"] = weather_weekly["week"].dt.start_time
weather_weekly = add_iso_week(weather_weekly)

In [138]:
# Norges Bank: daily -> weekly
exchange_rates_df["date"] = pd.to_datetime(
    exchange_rates_df["TIME_PERIOD"]
)

exchange_rates_df["OBS_VALUE"] = pd.to_numeric(
    exchange_rates_df["OBS_VALUE"],
    errors="coerce"
)

exchange_weekly = (
    exchange_rates_df
    .groupby(
        exchange_rates_df["date"].dt.to_period("W-SUN")
    )
    .agg({"OBS_VALUE": "mean"})
    .reset_index()
)

exchange_weekly["date"] = (
    exchange_rates_df
    .groupby(exchange_rates_df["date"].dt.to_period("W-SUN"))["date"]
    .first()
    .values
)

exchange_rates_df = add_iso_week(exchange_weekly)

In [139]:
daily_average_flows_entsoe_df = daily_average_flows_entsoe_df.drop(columns="date")
weather_weekly = weather_weekly.drop(columns=["date", "week"])
exchange_rates_df = exchange_rates_df.drop(columns="date")

In [140]:
# Load reservoir area polygons
reservoir_areas = gpd.read_file(
    "data/nve/Budområder/Budområder.shp"
).to_crs(epsg=4326)

# Convert weather coordinates to GeoDataFrame
weather_gdf = gpd.GeoDataFrame(
    weather_weekly.copy(),
    geometry=gpd.points_from_xy(
        weather_weekly["longitude"],
        weather_weekly["latitude"]
    ),
    crs="EPSG:4326"
)

# Spatial join: assign each coordinate to a reservoir area
weather_gdf = gpd.sjoin(
    weather_gdf,
    reservoir_areas[["budomr", "geometry"]],
    how="left",
    predicate="within"
)

# Convert back to regular DataFrame
weather_weekly = pd.DataFrame(
    weather_gdf.drop(columns=["geometry", "index_right"])
).rename(columns={"budomr": "area"})
weather_weekly["area"] = (
    weather_weekly["area"]
    .astype("string")
    .str.replace(" ", "_", regex=False)
)

# average in the area 
weather_weekly = (
    weather_weekly
    .groupby(["area", "iso_year", "iso_week"], as_index=False)
    [["temperature", "precipitation"]]
    .mean()
)

In [141]:
for name, df in {
    "Magasin": magasinstatistikk_df,
    "ENTSO-E": daily_average_flows_entsoe_df,
    "Weather": weather_weekly,
    "Exchange": exchange_rates_df
}.items():
    print(f"\n{name}, shape: {df.shape}:")
    print(df.head(4))


Magasin, shape: (260, 6):
   iso_year  iso_week  area  reservoir_level  level_TWh  capacity_TWh
1      2024         1  NO_4         0.527503  11.119345     21.079208
3      2024         1  NO_5         0.597728  10.394841     17.390587
5      2024         1  NO_2         0.657524  22.384480     34.043590
6      2024         1  NO_1         0.594819   3.570855      6.003264

ENTSO-E, shape: (106, 3):
      flow_mw  iso_year  iso_week
0  316.488444      2024        42
1  396.950804      2024        43
2  414.828923      2024        44
3  379.591970      2024        45

Weather, shape: (53, 5):
   area  iso_year  iso_week  temperature  precipitation
0  NO_1      2024         1   -16.577379         7.8036
1  NO_1      2024         2    -5.979449         5.9157
2  NO_1      2024         3   -14.830154        10.3622
3  NO_1      2024         4    -1.243873        29.8647

Exchange, shape: (151, 3):
   OBS_VALUE  iso_year  iso_week
0  10.814663      2024         1
1  10.821460      2024    

I can see that i need more power flow data, want entire 2024, so i will use api to retrieve more. I decided to let the code where i fetch from MongoDB database remain, because i assume using this database is implied part of assignment. 

In [ ]:
#todo 

In [142]:
def forecast_reservoir(
    magasinstatistikk_df,
    entsoe_df=None,
    weather_df=None,
    exchange_df=None,
    area="NO_1",
    history_weeks=156,
    horizon_weeks=12,
    lags=4,
    use_entsoe=True,
    use_weather=True,
    use_exchange=False,
    l2=0.01
):
    """
    Forecast weekly reservoir filling using River.

    Target: level_TWh for one selected reservoir area.

    External variables are held at their latest available
    values during recursive forecasting.

    Returns:
        forecast_df: Future predictions
        model: Trained River model
        training_df: Observations used for training
    """

    # --------------------------------------------------
    # 1. Prepare reservoir data
    # --------------------------------------------------

    target = magasinstatistikk_df[
        magasinstatistikk_df["area"] == area
    ][["iso_year", "iso_week", "level_TWh", "capacity_TWh"]].copy()

    capacity = target["capacity_TWh"].iloc[-1]

    # --------------------------------------------------
    # 2. Merge selected external data sources
    # --------------------------------------------------

    data = target.copy()
    exog_cols = []

    def merge_source(df, columns, prefix):
        nonlocal data, exog_cols

        if df is None:
            raise ValueError(
                f"{prefix} data was selected but not provided."
            )

        source = df[
            ["iso_year", "iso_week"] + columns
        ].copy()

        # Aggregate to one observation per ISO week.
        source = source.groupby(
            ["iso_year", "iso_week"],
            as_index=False
        )[columns].mean()

        source = source.rename(columns={
            col: f"{prefix}_{col}" for col in columns
        })

        new_cols = [
            f"{prefix}_{col}" for col in columns
        ]

        data = data.merge(
            source,
            on=["iso_year", "iso_week"],
            how="left"
        )

        # Fill missing observations using past values only.
        data[new_cols] = data[new_cols].ffill()

        exog_cols.extend(new_cols)

    if use_entsoe:
        merge_source(
            entsoe_df,
            ["flow_mw"],
            "entsoe"
        )

    if use_weather:
        merge_source(
            weather_df,
            ["temperature", "precipitation"],
            "weather"
        )

    if use_exchange:
        merge_source(
            exchange_df,
            ["OBS_VALUE"],
            "exchange"
        )

    data = data.sort_values("week_start").reset_index(drop=True)

    # --------------------------------------------------
    # 3. Create lagged features
    # --------------------------------------------------

    for lag in range(1, lags + 1):
        data[f"lag_{lag}"] = data["level_TWh"].shift(lag)

    feature_cols = (
        [f"lag_{lag}" for lag in range(1, lags + 1)]
        + exog_cols
    )

    # Select the requested training history.
    training_data = data.tail(history_weeks).copy()

    training_data = training_data.dropna(
        subset=feature_cols + ["level_TWh"]
    )

    if len(training_data) < max(10, lags + 1):
        raise ValueError(
            "Not enough valid observations for training."
        )

    # --------------------------------------------------
    # 4. Initialize and train River model
    # --------------------------------------------------

    model = (
        preprocessing.StandardScaler()
        | linear_model.LinearRegression(l2=l2)
    )

    for _, row in training_data.iterrows():

        x = {
            col: float(row[col])
            for col in feature_cols
        }

        y = float(row["level_TWh"])

        model.learn_one(x, y)

    # --------------------------------------------------
    # 5. Prepare recursive forecasting
    # --------------------------------------------------

    recent_levels = list(
        data["level_TWh"].dropna().tail(lags)
    )

    if len(recent_levels) < lags:
        raise ValueError("Not enough observations for lags.")

    # Latest available external observations.
    future_exog = {}

    if exog_cols:
        latest_exog = data[exog_cols].ffill().iloc[-1]

        if latest_exog.isna().any():
            raise ValueError(
                "Some selected external variables have no "
                "available observations."
            )

        future_exog = latest_exog.astype(float).to_dict()

    last_date = data["week_start"].max()

    predictions = []

    # --------------------------------------------------
    # 6. Recursive forecasting
    # --------------------------------------------------

    for step in range(1, horizon_weeks + 1):

        x = {
            f"lag_{lag}": float(recent_levels[-lag])
            for lag in range(1, lags + 1)
        }

        x.update(future_exog)

        prediction = model.predict_one(x)

        # Prevent physically impossible negative storage.
        prediction = np.clip(prediction, 0, capacity)

        predictions.append(prediction)

        # Feed prediction into the next forecast.
        recent_levels.append(prediction)

    # --------------------------------------------------
    # 7. Create forecast dataframe
    # --------------------------------------------------

    future_dates = [
        last_date + pd.Timedelta(weeks=i)
        for i in range(1, horizon_weeks + 1)
    ]

    forecast_df = pd.DataFrame({
        "week_start": future_dates,
        "forecast_TWh": predictions,
        "area": area
    })

    forecast_df["iso_year"] = (
        forecast_df["week_start"].dt.isocalendar().year
    )

    forecast_df["iso_week"] = (
        forecast_df["week_start"].dt.isocalendar().week
    )

    return forecast_df, model, training_data
